# ABS Business Indicators: pandas analysis guide

**Dataset:** ABS *Business Indicators, Australia* (dataflow `QBIS`), quarterly, **2023-Q1 to 2026-Q2**, six industries:
Accommodation & Food, Electricity/Gas/Water/Waste, Manufacturing, Mining, Retail, Wholesale.

**What you'll do**
1. Load and clean the file
2. Understand its structure (what's in it, and what isn't)
3. Build a helper to pull any single series
4. Analyse sales, profits, margins, real vs nominal growth, and wages
5. Check your numbers against ABS's own figures
6. Flag data-quality issues
7. Export a summary table

Run cells top to bottom with **Shift + Enter**.

## 1. Load the data
Click the folder icon in the left sidebar → upload `ABS BIS.csv`, or run the next cell and choose the file.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose 'ABS BIS.csv'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

raw = pd.read_csv("ABS BIS.csv")
print(raw.shape)   # (rows, columns)
raw.head(3)

## 2. Clean and understand the structure

ABS Data Explorer files come in **"long" format**: one row per observation, with a code column and a label column for each dimension.
We keep the readable labels, drop empty and duplicate columns, and convert `2023-Q1` into a proper quarterly date.

In [ ]:
df = raw.rename(columns={
    "Measure": "measure", "Price Adjustment": "price", "Industry": "industry",
    "Business Scope": "scope", "Adjustment Type": "adjustment",
    "TIME_PERIOD": "quarter", "OBS_VALUE": "value",
    "Unit of Measure": "unit", "Unit of Multiplier": "multiplier", "OBS_STATUS": "flag",
})[["measure", "price", "industry", "scope", "adjustment", "quarter", "value", "unit", "multiplier", "flag"]]

df["quarter"] = pd.PeriodIndex(df["quarter"], freq="Q")
df.info()

In [ ]:
# What's in each dimension?
for col in ["measure", "price", "industry", "scope", "adjustment", "unit"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().to_string())

In [ ]:
# Which measures exist for which business scope? (Shows gaps, e.g. Wages only exists for TOTAL)
pd.crosstab(df["measure"], df["scope"])

In [ ]:
# Which measures have real (Chain Volume) versions, and which have seasonally adjusted versions?
pd.crosstab(df["measure"], [df["price"], df["adjustment"]])

**Key things to notice**
- **Current Price** = nominal dollars. **Chain Volume Measures** = inflation-adjusted ("real"). These exist only for Sales and Inventories.
- **Seasonally Adjusted** removes regular quarterly patterns (e.g. Christmas retail). Use it for quarter-to-quarter comparisons. **Original** is the raw data. Ratios are only available as Original.
- **Scope**: TOTAL, CORP (companies), or UNINCORP (sole traders and partnerships).
- Dollar values are in **$ millions**.

## 3. Helper: pull one clean time series

In [ ]:
def series(measure, industry=None, price="Current Price", adjustment="Seasonally Adjusted", scope="TOTAL"):
    """Return one series (or one column per industry if industry=None), indexed by quarter."""
    q = df[(df.measure == measure) & (df.price == price) & (df.adjustment == adjustment) & (df.scope == scope)]
    if q.empty:
        raise ValueError("No data for that combination. Check the crosstabs in Section 2.")
    wide = q.pivot_table(index="quarter", columns="industry", values="value")
    return wide[industry] if industry else wide

sales = series("Sales")
sales.tail()

## 4. Analysis
### 4a. Latest-quarter snapshot by industry ($m, seasonally adjusted)

In [ ]:
latest = df["quarter"].max()
snapshot = pd.DataFrame({
    "Sales $m": series("Sales").loc[latest],
    "Gross operating profit $m": series("Gross Operating Profits").loc[latest],
    "Wages $m": series("Wages").loc[latest],
}).sort_values("Sales $m", ascending=False)
snapshot["Profit share of sales %"] = (snapshot["Gross operating profit $m"] / snapshot["Sales $m"] * 100).round(1)
print("Quarter:", latest)
snapshot.round(1)

### 4b. Sales trend by industry (real terms, inflation-adjusted)

In [ ]:
real_sales = series("Sales", price="Chain Volume Measures")
indexed = real_sales / real_sales.iloc[0] * 100          # rebase: 2023-Q1 = 100, so industries are comparable
ax = indexed.plot(figsize=(10, 4.5), title="Real sales, seasonally adjusted (2023-Q1 = 100)")
ax.axhline(100, color="grey", lw=0.8); ax.set_ylabel("Index"); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

### 4c. Growth rates: quarterly and year-on-year

In [ ]:
qoq = real_sales.pct_change() * 100        # quarter-on-quarter %
yoy = real_sales.pct_change(4) * 100       # same quarter last year %
pd.DataFrame({"Real sales q/q %": qoq.iloc[-1], "Real sales y/y %": yoy.iloc[-1]}).round(1).sort_values("Real sales y/y %")

### 4d. Economist's angle: how much sales growth is price vs volume?

Nominal growth ≈ price growth + volume growth. So **current-price growth minus chain-volume growth** gives an implied price change per industry.

In [ ]:
nominal_yoy = series("Sales").pct_change(4) * 100
real_yoy = series("Sales", price="Chain Volume Measures").pct_change(4) * 100
price_effect = (nominal_yoy - real_yoy).iloc[-1]
decomp = pd.DataFrame({"Nominal y/y %": nominal_yoy.iloc[-1], "Real (volume) y/y %": real_yoy.iloc[-1],
                       "Implied price y/y %": price_effect}).round(1).sort_values("Implied price y/y %")
decomp

In [ ]:
decomp[["Real (volume) y/y %", "Implied price y/y %"]].plot.barh(stacked=True, figsize=(9, 4),
    title=f"What drove nominal sales growth? (y/y to {latest})")
plt.axvline(0, color="black", lw=0.8); plt.grid(alpha=0.3); plt.show()

### 4e. Profit margins over time (ABS ratio, Original series, so expect seasonal swings)

In [ ]:
margin = series("Gross operating profit to sales ratio", adjustment="Original") * 100   # ABS stores this as a decimal (0.45); x100 for %
ax = margin.plot(figsize=(10, 4.5), marker="o", ms=3, title="Gross operating profit to sales ratio (%)")
ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()
margin.iloc[-1].sort_values(ascending=False).round(1)

### 4f. Wages: growth and labour intensity

In [ ]:
wages = series("Wages")
wage_yoy = (wages.pct_change(4) * 100).iloc[-1]
sales_to_wages = series("Sales to wages ratio", adjustment="Original").iloc[-1]   # $ of sales per $1 of wages
pd.DataFrame({"Wages y/y %": wage_yoy.round(1), "Sales per $1 of wages": sales_to_wages.round(1)}).sort_values("Sales per $1 of wages")

### 4g. Companies vs unincorporated businesses (gross operating profit)

In [ ]:
gop_corp = series("Gross Operating Profits", scope="CORP")
gop_uninc = series("Gross Operating Profits", scope="UNINCORP")
share_uninc = (gop_uninc / (gop_corp + gop_uninc) * 100).iloc[-1].round(1)
share_uninc.sort_values(ascending=False).rename("Unincorporated share of GOP %")

## 5. Check your numbers against ABS

The file includes ABS's own **quarterly % change** series. If your calculation from the levels matches theirs, your data handling is correct.
Get into this habit: it's the same discipline you'll use to evaluate an AI assistant later.

In [ ]:
mine = (series("Sales").pct_change() * 100).round(1)
abs_pct = series("Sales quarterly percentage change").round(1)
diff = (mine - abs_pct).abs()
print("Largest difference (percentage points):", diff.max().max())
diff.tail()

## 6. Data quality flags

Some observations carry a status flag (`x`, `y`, `z`). Look up each code's meaning in the [ABS Data Explorer](https://dataexplorer.abs.gov.au) (open the dataset and check the observation status notes). Treat flagged values with caution, and say so whenever you use them.

In [ ]:
flagged = df[df["flag"].notna()]
flagged.groupby(["flag", "measure", "scope", "industry"]).size().rename("observations")

## 7. Export your summary

In [ ]:
summary = snapshot.join(decomp).join(margin.iloc[-1].rename("GOP/sales ratio %"))
summary.round(1).to_csv("bis_summary.csv")
files.download("bis_summary.csv")
summary.round(1)

## Write up three findings

From the outputs above, write three one-sentence insights, e.g. *"Mining has the highest profit share of sales, but real sales fell X% y/y."* Check each number against the ABS release before you publish.

**Next:** when you get a Claude API key, these same `series()` calls become the tools Claude uses, and your Section 5 checks become its test set.